# 08. InfraScan Road Damage Detection Model Comparison & Evaluation

This notebook carries out the comparative evaluation and testing of trained YOLO models.

**Two-Phase Protocol:**
1. **Phase 1: Validation Split Model Comparison & Decision**
   - Both models (`yolov8s` vs `yolo11s`) are evaluated on the **validation split** (`conf=0.001`, `iou=0.6`).
   - The champion model is chosen strictly based on **validation mAP@50:95**.
   - The operational confidence threshold is chosen to maximize F1 on the **validation split only**.
2. **Phase 2: Final Test Split Evaluation (Protected Single Run)**
   - `test_once()` is executed exactly one time for each model on the unseen **test split**.
   - Protected by a single-run guard (`FORCE_RERUN=False`) to prevent test split data snooping.
   - Evaluates per-class metrics, per-country generalization, image-level FPR/FNR, 95% bootstrap CIs, 64-bit dHash leakage, and GPU inference speed.

### 1. Verify GPU Hardware and Environment

Check GPU availability for high-throughput benchmarking.

In [ ]:
import torch

print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name : {torch.cuda.get_device_name(0)}")

### 2. Mount Google Drive

Mount Google Drive to access trained checkpoints and save evaluation artifacts.

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")
assert os.path.exists("/content/drive/MyDrive"), "Google Drive mount failed!"
print("[OK] Google Drive mounted successfully at /content/drive.")

### 3. Install Ultralytics and Dependencies

Install the Ultralytics YOLO package.

In [ ]:
!pip install -q ultralytics

import ultralytics
print(f"[OK] Ultralytics version: {ultralytics.__version__}")

### 4. Extract Dataset and Code to Local Colab SSD (/content)

Extract evaluation dataset and detection code into `/content`.

In [ ]:
import zipfile
from pathlib import Path

code_zip_candidates = [
    Path("/content/drive/MyDrive/infrascan/ml_code_detection.zip"),
    Path("/content/drive/MyDrive/infrascan/data/ml_code_detection.zip"),
    Path("/content/drive/MyDrive/ml_code_detection.zip"),
]
code_zip = next((p for p in code_zip_candidates if p.exists()), None)
if code_zip:
    print(f"Extracting code from: {code_zip}...")
    with zipfile.ZipFile(code_zip, "r") as zf:
        zf.extractall("/content")

data_zip_candidates = [
    Path("/content/drive/MyDrive/infrascan/data/rdd2022_yolo_small.zip"),
    Path("/content/drive/MyDrive/infrascan/rdd2022_yolo_small.zip"),
    Path("/content/drive/MyDrive/rdd2022_yolo_small.zip"),
]
data_zip = next((p for p in data_zip_candidates if p.exists()), None)
data_extract_dir = Path("/content/rdd2022_yolo_small")
if not (data_extract_dir / "images").exists() and data_zip:
    print(f"Extracting dataset from: {data_zip}...")
    with zipfile.ZipFile(data_zip, "r") as zf:
        zf.extractall("/content")

print("[OK] Files extracted to /content.")

### 5. Setup Python Path and Import Evaluation Pipeline

Import the centralized configuration and evaluation suite.

In [ ]:
import sys
import json
import pandas as pd
from pathlib import Path

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

from ml.detection import config, evaluate_yolo
print("[OK] Evaluation modules imported successfully.")

### 6. Verify Model Checkpoint Paths on Google Drive

Locate the trained `best.pt` checkpoints for both YOLOv8s and YOLO11s.

In [ ]:
drive_models_dir = Path("/content/drive/MyDrive/infrascan/models/detection")
v8_best_pt = drive_models_dir / "yolov8s_rdd2022" / "weights" / "best.pt"
v11_best_pt = drive_models_dir / "yolo11s_rdd2022" / "weights" / "best.pt"

print(f"YOLOv8s best.pt exists: {v8_best_pt.exists()} ({v8_best_pt})")
print(f"YOLO11s best.pt exists: {v11_best_pt.exists()} ({v11_best_pt})")
assert v8_best_pt.exists(), f"Missing checkpoint: {v8_best_pt}"
assert v11_best_pt.exists(), f"Missing checkpoint: {v11_best_pt}"

### 7. Phase 1: Comparative Validation on the VALIDATION Split

We strictly evaluate both candidates on the **validation split** (`conf=0.001`, `iou=0.6`).
The test split is untouched. The superior model is selected based solely on validation mAP@50:95, and its optimal confidence threshold is chosen on the validation F1 curve.

In [ ]:
data_yaml = "/content/rdd2022_yolo_small/data.yaml"

print("Evaluating YOLOv8s on VALIDATION split...")
v8_val = evaluate_yolo.validate(v8_best_pt, data_yaml=data_yaml, split="val")

print("\nEvaluating YOLO11s on VALIDATION split...")
v11_val = evaluate_yolo.validate(v11_best_pt, data_yaml=data_yaml, split="val")

### 8. Validation Comparison Table & Best Model Selection

Compare validation metrics and pick the champion model based on validation mAP@50:95.

In [ ]:
val_comparison_df = pd.DataFrame([
    {
        "Model": "YOLOv8s",
        "Val mAP@50:95": v8_val["overall"]["map50_95"],
        "Val mAP@50": v8_val["overall"]["map50"],
        "Val Precision": v8_val["overall"]["precision"],
        "Val Recall": v8_val["overall"]["recall"],
        "Val F1 Score": v8_val["overall"]["f1"],
        "Optimal Conf Thresh": v8_val["chosen_threshold"],
    },
    {
        "Model": "YOLO11s",
        "Val mAP@50:95": v11_val["overall"]["map50_95"],
        "Val mAP@50": v11_val["overall"]["map50"],
        "Val Precision": v11_val["overall"]["precision"],
        "Val Recall": v11_val["overall"]["recall"],
        "Val F1 Score": v11_val["overall"]["f1"],
        "Optimal Conf Thresh": v11_val["chosen_threshold"],
    },
])

print(val_comparison_df.to_markdown(index=False))

# Programmatically determine champion
if v11_val["overall"]["map50_95"] > v8_val["overall"]["map50_95"]:
    champion_name = "YOLO11s"
    champion_pt = v11_best_pt
    champion_conf = v11_val["chosen_threshold"]
else:
    champion_name = "YOLOv8s"
    champion_pt = v8_best_pt
    champion_conf = v8_val["chosen_threshold"]

print(f"\n[DECISION] Champion Model Chosen (on Validation mAP@50:95): {champion_name}")
print(f"[DECISION] Operating Confidence Threshold (on Validation F1):   {champion_conf:.4f}")

### 9. Phase 2: Final Test Split Evaluation (Protected Single-Run Guard)

Now that model selection and threshold selection are frozen on the validation split, we run `test_once()` for **both** models exactly one time on the unseen **test split**.
The single-run guard prevents re-running test evaluation if `test_metrics.json` already exists unless `FORCE_RERUN=True`.

In [ ]:
# Safety guard flag
FORCE_RERUN = False

print("=== Testing YOLOv8s on TEST Split ===")
v8_test = evaluate_yolo.test_once(
    model_or_weights=v8_best_pt,
    data_yaml=data_yaml,
    conf=v8_val["chosen_threshold"],
    force_rerun=FORCE_RERUN,
)

print("\n=== Testing YOLO11s on TEST Split ===")
v11_test = evaluate_yolo.test_once(
    model_or_weights=v11_best_pt,
    data_yaml=data_yaml,
    conf=v11_val["chosen_threshold"],
    force_rerun=FORCE_RERUN,
)

### 10. Comprehensive Test Split Results Comparison Table

Display the final side-by-side comparison on the unseen test split.

In [ ]:
test_summary_df = pd.DataFrame([
    {
        "Model": "YOLOv8s",
        "mAP@50:95": v8_test["standard_mAP_metrics"]["mAP50_95"],
        "mAP@50": v8_test["standard_mAP_metrics"]["mAP50"],
        "Precision": v8_test["operating_box_metrics"]["precision"],
        "Recall": v8_test["operating_box_metrics"]["recall"],
        "F1 Score": v8_test["operating_box_metrics"]["f1"],
        "F1 95% CI": f"[{v8_test['bootstrap_95_ci']['f1']['ci_95'][0]}, {v8_test['bootstrap_95_ci']['f1']['ci_95'][1]}]",
        "Image FPR": f"{v8_test['image_level_metrics']['false_positive_rate_fpr']*100:.1f}%",
        "Image FNR": f"{v8_test['image_level_metrics']['false_negative_rate_fnr']*100:.1f}%",
        "Latency (ms)": v8_test["speed_and_size_benchmark"]["single_image_latency_ms"]["mean"],
        "Throughput (FPS)": v8_test["speed_and_size_benchmark"]["batch_throughput_fps"],
        "Model Size (MB)": v8_test["speed_and_size_benchmark"]["model_file_size_mb"],
    },
    {
        "Model": "YOLO11s",
        "mAP@50:95": v11_test["standard_mAP_metrics"]["mAP50_95"],
        "mAP@50": v11_test["standard_mAP_metrics"]["mAP50"],
        "Precision": v11_test["operating_box_metrics"]["precision"],
        "Recall": v11_test["operating_box_metrics"]["recall"],
        "F1 Score": v11_test["operating_box_metrics"]["f1"],
        "F1 95% CI": f"[{v11_test['bootstrap_95_ci']['f1']['ci_95'][0]}, {v11_test['bootstrap_95_ci']['f1']['ci_95'][1]}]",
        "Image FPR": f"{v11_test['image_level_metrics']['false_positive_rate_fpr']*100:.1f}%",
        "Image FNR": f"{v11_test['image_level_metrics']['false_negative_rate_fnr']*100:.1f}%",
        "Latency (ms)": v11_test["speed_and_size_benchmark"]["single_image_latency_ms"]["mean"],
        "Throughput (FPS)": v11_test["speed_and_size_benchmark"]["batch_throughput_fps"],
        "Model Size (MB)": v11_test["speed_and_size_benchmark"]["model_file_size_mb"],
    },
])

print(test_summary_df.to_markdown(index=False))

### 11. Per-Class and Per-Country Performance Breakdown

Inspect per-class detection accuracy and cross-country geographic generalization for the selected champion model.

In [ ]:
active_test = v11_test if champion_name == "YOLO11s" else v8_test

print(f"=== {champion_name} Per-Class Metrics on Test Split ===")
class_df = pd.DataFrame(active_test["per_class_metrics"]).T
print(class_df[["precision", "recall", "f1", "tp_boxes", "fp_boxes", "fn_boxes"]].to_markdown())

print(f"\n=== {champion_name} Per-Country Generalization on Test Split ===")
country_df = pd.DataFrame(active_test["per_country_metrics"]).T
print(country_df[["total_images", "precision", "recall", "f1", "tp_boxes", "fp_boxes", "fn_boxes"]].to_markdown())

### 12. Near-Duplicate Leakage Audit Findings

Inspect the 64-bit dHash train/test duplicate leakage check to evaluate whether consecutive road video frames biased test metrics.

In [ ]:
leak_info = active_test["near_duplicate_leakage_audit"]
print(f"Test Images Audited          : {leak_info['test_image_count']}")
print(f"Near-Duplicates (Hamming <= 2): {leak_info['test_images_with_near_duplicate']} ({leak_info['near_duplicate_percentage']} %)")
if 'with_near_duplicate' in leak_info and 'without_near_duplicate' in leak_info:
    print(f"  Performance on Duplicate Subset    : F1 = {leak_info['with_near_duplicate']['f1']} (count={leak_info['with_near_duplicate']['count']})")
    print(f"  Performance on Non-Duplicate Subset: F1 = {leak_info['without_near_duplicate']['f1']} (count={leak_info['without_near_duplicate']['count']})")

### 13. Qualitative Predictions and Error Diagnostics

Display qualitative bounding box predictions (8 good, 8 false negatives, 8 false positives) saved during test evaluation.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

eval_dir = drive_models_dir / ("yolo11s_rdd2022" if champion_name == "YOLO11s" else "yolov8s_rdd2022") / "evaluation"
vis_dir = eval_dir / "qualitative_examples"

if vis_dir.exists():
    sample_images = sorted(list(vis_dir.glob("*.jpg")))[:6]
    if sample_images:
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        for ax, img_p in zip(axes.flat, sample_images):
            img = Image.open(img_p)
            ax.imshow(img)
            ax.set_title(img_p.stem, fontsize=9)
            ax.axis("off")
        plt.tight_layout()
        plt.show()
else:
    print(f"Qualitative directory not yet generated at: {vis_dir}")

### 14. Plain-Language Executive Summary

**Key Evaluation Conclusions:**
1. **Model Selection:** The champion architecture was chosen strictly on the **validation split** (by validation mAP@50:95) to preserve complete scientific integrity.
2. **Operational Threshold:** The confidence threshold was optimized on validation F1 curve, avoiding arbitrary 0.50 defaults.
3. **Test Split Generalization:** Both architectures were tested once on the unseen test split without post-hoc data snooping.
4. **Cross-Country Generalization:** The per-country breakdown highlights variation across road surfaces, lighting, and sensor viewpoints.
5. **Inference Latency & SLA:** GPU inference latency (batch 1) and batch-16 throughput confirm readiness for real-time edge or cloud inspection pipelines.